# Бросаем мячик

## Что интересного, что используется

- обычная школьная физика траектории полёта тела, брошенного под углом к горизонту. matplotlib выводит график
- просто для оформления и ради фана выводятся красивые формулы с помощью Latex
- анимация для каждой точки траектории. В качестве маркера анимации используется произвольное изображение
- файл изображения зашит в тело программы в base64, можно не использовать внешний
- анализируется среда запуска - jupiter notebook или обычная. Выбирается нужный метод отображения анимации
- явное использование display в notebook

In [18]:
from math import sin, cos, radians
import io
import base64

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.animation import FuncAnimation

Расчёт траектории полёта мяча. Входные данные угол броска $\alpha$ и скорость $v_0$. `num-ticks` - количество кадров анимации.

In [19]:
g = 9.8

num_ticks = 50  # кадров анимации
alfa = radians(45)  # угол броска
v0 = 10 # скорость броска

In [20]:
def get_img_ball(file_name: str = None):
    """ Получить иконку анимации из файла (png 50×50) или взять вшитую по умолчанию """
    if file_name:
        img_ball = plt.imread(file_name)
    else:
        image_base64 = b"iVBORw0KGgoAAAANSUhEUgAAADIAAAAyCAYAAAAeP4ixAAAABGdBTUEAALGPC/xhBQAACklpQ0NQc1JHQiBJRUM2MTk2Ni0yLjEAAEiJnVN3WJP3Fj7f92UPVkLY8LGXbIEAIiOsCMgQWaIQkgBhhBASQMWFiApWFBURnEhVxILVCkidiOKgKLhnQYqIWotVXDjuH9yntX167+3t+9f7vOec5/zOec8PgBESJpHmomoAOVKFPDrYH49PSMTJvYACFUjgBCAQ5svCZwXFAADwA3l4fnSwP/wBr28AAgBw1S4kEsfh/4O6UCZXACCRAOAiEucLAZBSAMguVMgUAMgYALBTs2QKAJQAAGx5fEIiAKoNAOz0ST4FANipk9wXANiiHKkIAI0BAJkoRyQCQLsAYFWBUiwCwMIAoKxAIi4EwK4BgFm2MkcCgL0FAHaOWJAPQGAAgJlCLMwAIDgCAEMeE80DIEwDoDDSv+CpX3CFuEgBAMDLlc2XS9IzFLiV0Bp38vDg4iHiwmyxQmEXKRBmCeQinJebIxNI5wNMzgwAABr50cH+OD+Q5+bk4eZm52zv9MWi/mvwbyI+IfHf/ryMAgQAEE7P79pf5eXWA3DHAbB1v2upWwDaVgBo3/ldM9sJoFoK0Hr5i3k4/EAenqFQyDwdHAoLC+0lYqG9MOOLPv8z4W/gi372/EAe/tt68ABxmkCZrcCjg/1xYW52rlKO58sEQjFu9+cj/seFf/2OKdHiNLFcLBWK8ViJuFAiTcd5uVKRRCHJleIS6X8y8R+W/QmTdw0ArIZPwE62B7XLbMB+7gECiw5Y0nYAQH7zLYwaC5EAEGc0Mnn3AACTv/mPQCsBAM2XpOMAALzoGFyolBdMxggAAESggSqwQQcMwRSswA6cwR28wBcCYQZEQAwkwDwQQgbkgBwKoRiWQRlUwDrYBLWwAxqgEZrhELTBMTgN5+ASXIHrcBcGYBiewhi8hgkEQcgIE2EhOogRYo7YIs4IF5mOBCJhSDSSgKQg6YgUUSLFyHKkAqlCapFdSCPyLXIUOY1cQPqQ28ggMor8irxHMZSBslED1AJ1QLmoHxqKxqBz0XQ0D12AlqJr0Rq0Hj2AtqKn0UvodXQAfYqOY4DRMQ5mjNlhXIyHRWCJWBomxxZj5Vg1Vo81Yx1YN3YVG8CeYe8IJAKLgBPsCF6EEMJsgpCQR1hMWEOoJewjtBK6CFcJg4Qxwicik6hPtCV6EvnEeGI6sZBYRqwm7iEeIZ4lXicOE1+TSCQOyZLkTgohJZAySQtJa0jbSC2kU6Q+0hBpnEwm65Btyd7kCLKArCCXkbeQD5BPkvvJw+S3FDrFiOJMCaIkUqSUEko1ZT/lBKWfMkKZoKpRzame1AiqiDqfWkltoHZQL1OHqRM0dZolzZsWQ8ukLaPV0JppZ2n3aC/pdLoJ3YMeRZfQl9Jr6Afp5+mD9HcMDYYNg8dIYigZaxl7GacYtxkvmUymBdOXmchUMNcyG5lnmA+Yb1VYKvYqfBWRyhKVOpVWlX6V56pUVXNVP9V5qgtUq1UPq15WfaZGVbNQ46kJ1Bar1akdVbupNq7OUndSj1DPUV+jvl/9gvpjDbKGhUaghkijVGO3xhmNIRbGMmXxWELWclYD6yxrmE1iW7L57Ex2Bfsbdi97TFNDc6pmrGaRZp3mcc0BDsax4PA52ZxKziHODc57LQMtPy2x1mqtZq1+rTfaetq+2mLtcu0W7eva73VwnUCdLJ31Om0693UJuja6UbqFutt1z+o+02PreekJ9cr1Dund0Uf1bfSj9Rfq79bv0R83MDQINpAZbDE4Y/DMkGPoa5hpuNHwhOGoEctoupHEaKPRSaMnuCbuh2fjNXgXPmasbxxirDTeZdxrPGFiaTLbpMSkxeS+Kc2Ua5pmutG003TMzMgs3KzYrMnsjjnVnGueYb7ZvNv8jYWlRZzFSos2i8eW2pZ8ywWWTZb3rJhWPlZ5VvVW16xJ1lzrLOtt1ldsUBtXmwybOpvLtqitm63Edptt3xTiFI8p0in1U27aMez87ArsmuwG7Tn2YfYl9m32zx3MHBId1jt0O3xydHXMdmxwvOuk4TTDqcSpw+lXZxtnoXOd8zUXpkuQyxKXdpcXU22niqdun3rLleUa7rrStdP1o5u7m9yt2W3U3cw9xX2r+00umxvJXcM970H08PdY4nHM452nm6fC85DnL152Xlle+70eT7OcJp7WMG3I28Rb4L3Le2A6Pj1l+s7pAz7GPgKfep+Hvqa+It89viN+1n6Zfgf8nvs7+sv9j/i/4XnyFvFOBWABwQHlAb2BGoGzA2sDHwSZBKUHNQWNBbsGLww+FUIMCQ1ZH3KTb8AX8hv5YzPcZyya0RXKCJ0VWhv6MMwmTB7WEY6GzwjfEH5vpvlM6cy2CIjgR2yIuB9pGZkX+X0UKSoyqi7qUbRTdHF09yzWrORZ+2e9jvGPqYy5O9tqtnJ2Z6xqbFJsY+ybuIC4qriBeIf4RfGXEnQTJAntieTE2MQ9ieNzAudsmjOc5JpUlnRjruXcorkX5unOy553PFk1WZB8OIWYEpeyP+WDIEJQLxhP5aduTR0T8oSbhU9FvqKNolGxt7hKPJLmnVaV9jjdO31D+miGT0Z1xjMJT1IreZEZkrkj801WRNberM/ZcdktOZSclJyjUg1plrQr1zC3KLdPZisrkw3keeZtyhuTh8r35CP5c/PbFWyFTNGjtFKuUA4WTC+oK3hbGFt4uEi9SFrUM99m/ur5IwuCFny9kLBQuLCz2Lh4WfHgIr9FuxYji1MXdy4xXVK6ZHhp8NJ9y2jLspb9UOJYUlXyannc8o5Sg9KlpUMrglc0lamUycturvRauWMVYZVkVe9ql9VbVn8qF5VfrHCsqK74sEa45uJXTl/VfPV5bdra3kq3yu3rSOuk626s91m/r0q9akHV0IbwDa0b8Y3lG19tSt50oXpq9Y7NtM3KzQM1YTXtW8y2rNvyoTaj9nqdf13LVv2tq7e+2Sba1r/dd3vzDoMdFTve75TsvLUreFdrvUV99W7S7oLdjxpiG7q/5n7duEd3T8Wej3ulewf2Re/ranRvbNyvv7+yCW1SNo0eSDpw5ZuAb9qb7Zp3tXBaKg7CQeXBJ9+mfHvjUOihzsPcw83fmX+39QjrSHkr0jq/dawto22gPaG97+iMo50dXh1Hvrf/fu8x42N1xzWPV56gnSg98fnkgpPjp2Snnp1OPz3Umdx590z8mWtdUV29Z0PPnj8XdO5Mt1/3yfPe549d8Lxw9CL3Ytslt0utPa49R35w/eFIr1tv62X3y+1XPK509E3rO9Hv03/6asDVc9f41y5dn3m978bsG7duJt0cuCW69fh29u0XdwruTNxdeo94r/y+2v3qB/oP6n+0/rFlwG3g+GDAYM/DWQ/vDgmHnv6U/9OH4dJHzEfVI0YjjY+dHx8bDRq98mTOk+GnsqcTz8p+Vv9563Or59/94vtLz1j82PAL+YvPv655qfNy76uprzrHI8cfvM55PfGm/K3O233vuO+638e9H5ko/ED+UPPR+mPHp9BP9z7nfP78L/eE8/stRzjPAAAAIGNIUk0AAHomAACAhAAA+gAAAIDoAAB1MAAA6mAAADqYAAAXcJy6UTwAAAAJcEhZcwAACxMAAAsTAQCanBgAABSoSURBVGiB3Zp5lFT1mfc/d6t97ba76JaG7ga6UVCJQgRBEXJiTJgMRo0zGpd4Mp53Bowal7hM3nmd5CTMezyJLyYZScQoi9GcMcElk5AYBMEBAqgIHpuWpquB3qq7qK59uev7R9W9dAHx1cx/7++ce+rWrV/f+3yf/fneFizL4v+HJX/SjZZlYVkWoihiWRa6rmOaJpZlIcsysiwDtFiW9RnTNC+zLKvDsqwppmWFsSwfoAmCkBMEISlJ0kA6nd6RSCSOhEKhAU3TLFXTKJZKmEDE58eQYWhwGK+sUNE0gn4/M2bOJBQK/feAnGuJooiiKF7g66qq3lIulxcKgiBLkoTX60UQBABM03SUYZomCAKyLH/7z3v30t7efqK9o+NtXddf9/p8r6iaVhYQgE/nKX81EEVRlgCrDhw4sKyvr2+Kz+fD6/UiiiKRSBi/38+HH35IW9s05s+fj2VZCIKAKIoIQCgUolgs8vAjj0zr7uq65Y6v33FLJBodrVS055s7Op4qmZWRT+P24qcRXhRFdN3oEAThl5lMZtcvX/jlzTt27JhSKBQwTZNyuYwgCHz4YQ+rV6/mhhtu5MiRHgRBwDAMDMPANE10XUfXdVatXs3atWsxTJP777+fzZs2TSkXC4/09vb2lYvlR0PBEJ8UjPBpUJdKpX/0er1PxQcGlK1btyIJIhdceAGVSoVsNovH46H/2DF+9OSTxONxrrzySnbu3IllWaiq6rga4FhHkiQANm3axOOPP87555/PNZ//PJ2dM+ju7joQaWi4ZXh4+KjP42FWV9dfjJFPCkTI5XK/CgQCX+3r6+ONP/2J5qYmGhoaSKVS6LqOKEmMDg/zf9auJR6PI8sSr732Gl/84peoVCqO8JM/7SXLMoIgkEqlmDtnLiOjI1w0dy4NDQ3ccsvXylctveoGt9v9u2g0SiQS+auBNKZSqT+FwuF5I8PDbP3972lqbkKSFAqFPJqmOdnrhc2b+dO2bQBcfvln2bZtG35/wHG5vwTE/i7LMoODQ7z22quEgkHefe89Pur9CH8gwKpV//T1pUuXbvhLQn5ssFuW1ZhKpd4NBALTKuUyu3btJBKNYhgm+Xza8XWfz0fvkSO8+eb2KvLGRhYtugKPx1sVVBSxaplLEAQn8E8/BwQBdF1n6tTzWbVqFQC33nYbhUKBX/ziOZ577rnnzzvvvKvnzJlz57lk/TiLCBPpiSMCQpfL5eLVV7agahrBYIhcLoeu62iaRigUJJvLsfbJtRRLRb55zz186YtfJBaLOfWmoaGhlih0RFGst4pQTbb2NVEUUVUNWZaqGa62L5FIkM1m6ejo3CTL0u2fGEg2m/1DpVK5xu12IwgC+/fvZ/++/TQ0NqC4FErFEg0NUQ4dOsyOt3ay+IoruPvu1UybNs25h6qq5HI5yuUy0WgUj8eDpml1LnauBKCqGooiI4qi47Z2UlBVDZdLuRv46f8TiGXxaDI5/gOPx4vb7cLlcgFw+PBhXn75ZaLRKK2trbz++utkMlnuvvturrnm8wDYqViSJARBwOPxUCgUSKVShMNhQqEQlYqKKAoIgogongZlW6tSqSArClLtu2VZGIaBZYGiSOzYsZOWltaru7tnvVUHZGxsDLArsNX15717e69etpxwOAxUfVcQBCRJoq+vj40bN7J3715mzepizZofOMXN3ucUvlo8uN1udF1neHiYSCRCNBo9pxd83LLByLLMnj17ePbZZ7Xvfve7zaIoph0g+/btwzAMGhoa+M2vX+5RZGX2tx54AEmSnCJmW87lcrF3z17eeOOP/M9/+RcAMplMnYuIouho1xbCTrHJ8STJU0mOHTvGyZMn8ft9gIAoiCBYKIpCZ+cMrrjiinOCsa195513Iori75544okV+Xy+mrWCwSCiKNLf33/dn/ftm71hw8azQNhHqVTioosvYuGihaiqSqFQcBpJ20XsvZPPS6USkUiEdCbNN+++G7fbTffsbhoaGnG5qu4ryxIgMDIyiiCILFq0sA7IZGWtXr2af7jrri8dOnRoka7re2QAwzAIBoNs+c1v/vfChYsIh8MO+smfoihSKpUoFArVjKRpk1zy9MNsYJMPo7bno94jjI2Ncfsdd3DhhRcyMTExSQkisiwhSxL79+8jEolwwQWz68DYlp4/fz6hUIgNGzb+aPXqVYtEqPaZA/GB+QMDA11/+7dfxgZnmmbdYVkWxWLJ0bRhmlhW/e/2YV8zDKOadgUBLHj//UPV3wyTRCJBoVCgUqmgaRq6rqFpGqqqEg6HefPNbYyPJ8+yig38swsWcPDgewunTJnSJQK4FIWtW7f+o8/n48IL5zhannwYhoGmaWiaWqsJWq0RPBvsZAC6rmMYBv5AgOHhYY4cOcLwyAjxeD9er9fZZyvOMAx0w0CSJBRF5j//87fohnFWvNhWicfjHDhw4C4RwO/388EHh75y6aWXngVictdaqVQwDANRFKlU1Dphz7TAmYdLUThx4ji9vUfI5/Ps3LmT0dFRXIriPMMGZBgGqqoSCAQZHxtjx/Yd54yVaDRKJpNh9+7d14kA77777gIEGpYtWwZUC9mZglmWiSAI+Hw+gJornNa4fa6qat1hKwGg50gPI6MJFEXhWH8/O7ZvR5JlrJor24dlWVg1xbW1tbF7925GhkcAKJVKZHM5x/0Beo/0zhQBTqWSS1KnUrjcbscijob0qgU8Hi/FYonjxwdwu1yOVWyfrlQqlMtlKpUKlUoFVVUdcLKiMJFKcfC996hUKng8HrxeD++8+y7xY/143O4a4NNgjFocVadJiWee+TnZbJZKpYJUq/jXXnstyz/3OQ68c6A6WKVSqYtcLg9NTU2On9u+7/V5kSSJeH+cFzZv5sEHH2TDxo2Ew2G8XjelUolyuVy1QA2UYym9qrHmpmb27NnDiy++hK7rNSA+0uk023dsp1QqIssyuqbXubNu6BSLRaLRKAfeeQfDMIlEIvj9fqdv+9EPf0hjY2O1joyNjnfe/8ADdHZ2kkqlcLlciKKAzxdgYiLNwYMHWb9+Pa+9+hq6oXPsWD+lUok77riDQCBAKpVCkqS6aq4oCqIoUsjnyedzHD16lPFkEkVR8Pv8mJaJz+fj4PsHaW9v5+plyxwl2DFg3yeRSHD++ecTiYQd17c955JLLuH6r3wF6fHHHyfe3//tldetbCqXyxQKBTRN47zzzuPEyZO8+uprrF27ljfeeAPDrGo4n8/z9q63KRaLXH755QSDQbLZLKIgotSCN51Ok8/nERAwLZOlV1+Ny+Vi27ZtlCtl3G43sixTLlcYGxtjdnc3jY2NlMvlujbf7/dz+PBhuru6uWLxFWi12mXHiCzLjCeTVSCFQuFbM2fObEin0wiCgOJysXv3bjY8/zwvvPAChw4dQhQEPF6v0wyqmsq+ffvo7+9nwYL5tE2dysjoCPl83vHjQDBIJBJxEseKFSuQJYlt27ZV90gSbrebU6dOkcvlmNXVRVNTEx6PB0VRkGWJYDBMT08PVy29iq6urrruGUCSJHw+XxVIIBi8HWi1U6sgCPz7T3/K+vXrGRsfr7YPilLXetgt9kcffcQf//BHWltb6OycgWVZhEIhwpEIAjgaLpfLaJrGihUrqFQq7Nq1C4vqvVwuF0NDQ/T29pJIJMhlsxSLRTRNIzk+TiqV4rbbbyMYDGIYxhmTpkBjY0M1Rnw+n2i337Isk8vlOJVKEY5EyeWyDnFgV1T7U5ZldF1n4Phxeo708nd/fzPxeBxd18nncrW4kWoPFUmn04iiyJo1a0hPTLDuZz/DpVRHBLfbRTweJx6PIwoigWAAt9vN+Pg4jz36GK0trXVudbrLrt5fpKo100Zpj67BYBC/30dLS4tz3bIszEmV3DKrgKZMmcLKlSs5OThYtQDWpOnu9LwjyzKpVIpUKsXT69Zx4w03kM/nCQQDeL0+IpEIbrcb0zKplMt4PR6WLVvGzbfcXIuJ02OCDUIQqqOyXNNw2k67uq4TiURoamqmVCoRi8WYEosxPDKCaZq4FBeSPYYikMvnWLx4Me3tHYyOjlSFt5jUCNr+bAECiqIwNjaGx+Nh0+bNJJNJdrz1FpFIhOnt05k1cxZz587l4osuZs6cC+nq7iabzZLP5/F6vY5rn3YvsCyzCkQUxROqqjp9kizLtLa2OHN2lXAwKBZLuD1uJElyBM3lc1x80UW43a5zkApW3bUqyVAFMzh4klmzulj3s5+xbt06lixZwpy5c+ho78BdK8z2KpVKAAQCAaea2yCqadhyLHLYbiNM00TTNKZOnVpXeCLRKF6fr25G0TSNcCRM9+zZVCqVOrPXrbrvFkItTQ8NDRGLNfPkk086wAuFIqVSyRnOJEkiHA6TTqdRVRVZls9iH03TqsaIoihvT269C4UCbVPbaG5udoqPTQCcfoBIpVKho72Dzs5O8vm8ExdncliCE5yTXU1AlmVGRkYZGhpyYkfT1Lq/tSyrNnTJFItFZx6ZbGldN6pAfD7fflEUx+0utlAo0Hp+K9OnT3cmwKogIpIs1V4jKKiaSkdHB7FYDFVV6+ibs+OjKrwthNvjIZfLYZomgUDgrOFsMhjDMBwGxjRNx1KZTIYtW7YgikJFhCprIYriK7bbqKqK3+/n0ksvZWxsjJMnT5JMJslk0hTyBYrFIvl8nlKxREtLC9FaDP0laslygv/073aNCQaDGLp+xn6r7lzXddxuNx6vx1HW0aNHufPOO9m+fTtut+tNGcDr9eL1ev9dEMW7JmpcbjKZZOHChTz88MOMjo4yMTFBoVCgVCrVJj6R5qYmLrnkEofbPRuAdVb2Mk0Tr9dLJpPGsiwCgSC6rp1lzTPvI8syRsHgvffe44MPPuCll36FJIlceeVVqJr2jAywdet2yuXyQSzj3UWL51+ansiQTqcBuPfeexEEgWw2S6lUqhuugsEg6XSanp4eGhoa6gqWDcB2BTsGBUHANE2nq9V1rW5EBjAtC6G2fzIz+Ztf/5ofrFnDjBmdNDfHaGtrw+VSejLp9BYZ4OjRX2NZJsPD+aczmcwzK/7mC05bMTY+DrVAd7lceL3eOkG9Xi/j4+NkMhnC4bBTOCd3AlAlDbxeL62trZw8eQLDMPH5/GjaaWuYpolYiw/TtBDFqiL8fj+6prFhw0ZcbhfNzTGCwSDBYJDW1pbbBwcHq+n3D3/YjyAIaJrw7N49g08MD5+IXLV0KYFAABEBtTafn7ls0i4Wi5FIJDBNiwsumO0IVW38qm1MpVIhmUyyedMmmpqbWLLkSqf9t+8lCEKV8LYswMQ0T5N9L2zezAcffMAXrv2CE+yxWOx9VdUO5HL5KkH38MP/q0pHukQCfuW23//u9xvj8QH+9bv/ysqVK+nr66t74GQgUO1AJUkilUrx9q5dyIqCx+0hl8uSyWYp5POcHBykr6+Pnp4eli9bxuu//S3lcplMJoMsy4iSiCiIzr3se4dCIXRdZ/ny5VTKFbq6unB73EyfPp1bb711sSRJu6HWorS0RCdniE0Xzplz/6HDh+dt2LCB5cuX4/F4KJVKdTVisuuYpklzczP9/f3883e+gyRJeDxeVLVK89hvff1+Px0dHezZs4fvfe97fP/736dYKGKY1R7KEi1nXqcWG7Is85Of/IQPez7kc8s/h0U18Lu6ul7XdX23HZcywK5du+pcZsqUKddff/31/S+++CLPPvss9957Lz09PU4xPFfRUxSFvXv3IkkS06dPd+JicgEDEAWB5liMp59+mrlz5/K1r32N/v5+BMFEEMA0a1YWBCKRCO+88w7r1q1jdvdsNE1DkiQWLFigrly58tbJyUUGmD37NJtXq+DxQCDwjWuuuebZp556iosvvqjGIQ3URthqcZwsbC6Xo7e3t1r9BaFW+6y6V9MCoEO1Jng8fOc7/0xHezuXL1zIwMAAouhxLExN85s3b2ZoaIiZM2ciiiJdXV3MmDHj2mPHjmUnx61s++HkVWPQf9HZ2Tm7p6fnoccefYznnn+e5uZmRkdHkGXFAWC3EIlEghMnjuPz+TAtC2oNooXFmfXdMAwikQiJRIJv3X8/69evJxaLkc/nnRiBakzOu2QeoVAI0zSZM2cOfX19q15++eXtZxLnMsAjjzzCuVZ7e/u3r7vuusCWLVv+6b577+PHP/kx0WiUZPJUNUBrYDweD319fQwPDTutdp1i7JNJb6cMw6C1tZV9+/bxxBNP8NRTT9X9jU0IzvvMPGKxGKFQCE3THnzllVeePlcB/tj37DWuatXSpUvXjIyOcM8995BMnkJRFCpqxemEfV4fBw8eZHhkBLfbdRaFOsnUdZV+YmKCUCjE8hoxaI8HNhVrt+8zZ87Asqxv9Pb2/jAYDJ5T1o8FUvv/EsrlymOXXXbZXUePHmXz5s2EQiFKxRITEylOnTpFPN7PeY2NNDU1MTBwnEKhUJP7NKFd3z9VKZ2RkRFWrVrFV2+6ibGxMadrqHJbevUFlCB89I1v/MNiRVF+oaqqUwY+FRB7qapKuVxe/9nPLrhk7569e1966SUURUFVNcrlMoNDQ8z7zGd45uc/56abbgLg+PHjJBIJSqVSHbmNZWGaOsePH+fGG2/kvvvuI5FIUC6XHXYzn8+TTCYB1nR3dV0wderU3ecqyHVK/yRA7JohCsKhtmlti9588807dF3/5rx58y6z55TBwUFaWlp46MGHWLFiBXv27OH9999nIB4nkUhU64kkYdSYxC9/+cus+bd/o1AoMDGRQnG5KOTzFEpFXIrrv8Lh8KOxWGyXLMtkMhlHjv8WEHtVJ8IIfr9/w+7duzcUisWbp7W1PRAIBC7zer0cHzjOsGuYpqYmbr/9drLZLCdPnCAejzM4NETqVIpiqUjbtDYeeuAhopEIoyMjCAgMnjgJgrArGo3+uLm5+T88Ho9DvX4cgL8KiCAIzugryzKnkskXDV1/MZvN/t2UKbH/EY1Gl/l8ftLptNNgNjY10d7RgSgIaJqOJIlEohH27f8zJ46fRFGU0cHE4OvdM7tfWLJ4yVuRSARRFNE0DY/H84ll+78pNxKTLN2hBgAAAABJRU5ErkJggg=="
        img_ball = plt.imread(io.BytesIO(base64.b64decode(image_base64)))
    return img_ball


def update_animation(frame):
    """ На каждом кадре анимации меняется положение иконки мяча """
    box_ball.xybox = x[frame], y[frame]
    return box_ball,


def is_jupyter_notebook() -> bool:
    """ Проверка среды выполнения """
    try:
        from IPython import get_ipython
        if 'IPKernelApp' in get_ipython().config:
            return True
        return False
    except Exception:
        return False

Просто для демонстрации применяется Latex дла отображения формул в markdown

$$x(t) = v_0 \cdot \cos(\alpha) \cdot t$$
$$y(t) = v_0 \cdot \sin(\alpha) \cdot t - \frac{1}{2} gt^2$$

Координаты определяются как функция времени

In [21]:
t_full = 2 * v0 * sin(alfa) / g # Полное время полёта до земли
ticks = np.linspace(start=0, stop=t_full, num=num_ticks, endpoint=True)

x = v0 * cos(alfa) * ticks
y = v0 * sin(alfa) * ticks - 0.5 * g * ticks ** 2

In [26]:
fig, ax = plt.subplots()
ax.plot(x, y, alpha=0.5, ls=':')    # основной график - параболическая траектория
# Для анимации будем вместо встроенных маркеров будем использовать иконку png 50×50 пикселей с альфа-каналом
image_box = OffsetImage(get_img_ball(), zoom=0.3)  # zoom подбирается. 50 пикселей слишком много, уменьшим
box_ball = AnnotationBbox(image_box, (x[0], y[0]), frameon=False)   # положение будет менять функция анимации
ax.add_artist(box_ball)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.axis('equal')
ax.set_title("Ball fly")

# создаём анимацию, каждый кадр формирует update_animation
ani = FuncAnimation(fig, update_animation, frames=num_ticks, interval=t_full/num_ticks*1000)
if is_jupyter_notebook():   # и отображаем её
    from IPython.display import HTML, display   # display можно не импортировать
    plt.close(fig)  # чтобы не появлялось статическое окно после окна анимации
    display(HTML(ani.to_jshtml()))  # без явного display окно анимации не выводится
else:
    plt.show()